In [186]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers

Datasets = []
PREDICTORS = ["WdRef", "WeRef"]   
PHYSICAL_PREDICTORS = ["Wd", "We"]   
TARGET_INT = ["Theta"]  
TARGET = ["DeltaTheta"]
     
TIME_STEPS = 3
TS = 0.07
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)  

In [187]:
for i in range(4):
    Dataset = pd.read_excel(f"../../../RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(2):   
    Dataset = pd.read_csv(f"../../../Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS

    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset 

In [188]:
NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

for i in range(5):
      CurrentTestDataset = Datasets[i + 1]
      CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
      CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
      NormDatasets.append(CurrentTestDataset)

In [189]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

In [190]:
x_train, y_train = CreateSequences(Train1[PREDICTORS], Train1[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(x_val)}")

Dimensão da entrada: (973, 3, 2)
Dimensão da saida: (973, 1)
Dimensão da entrada: (1268, 3, 2)
Dimensão da saida: (1268, 3, 2)


In [191]:
Wd_train =  Train1["Wd"].values[:len(x_train)]
We_train =  Train1["We"].values[:len(x_train)]

In [192]:
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada fisica : {np.shape(Wd_train)}")
print(f"Dimensão da entrada fisica: {np.shape(We_train)}")

Dimensão da entrada: (973, 3, 2)
Dimensão da saida: (973, 1)
Dimensão da entrada fisica : (973,)
Dimensão da entrada fisica: (973,)


In [193]:
import matplotlib.pyplot as plt
import numpy as np

def PlotOut(ax, title, target_name, y_true, y_cin, y_pred):

    time = (np.arange(0, len(y_pred), 1).astype(float) * 0.07).round(5)

    ax.scatter(time, y_true,
               marker='o',
               s=12,
               color='tab:blue',
               label='Amostras Reais',
               alpha=0.7)

    ax.scatter(time, y_pred,
               marker='x',
               s=12,
               color='tab:red',
               label='Modelo PIRNN',
               alpha=0.7)

    ax.plot(time, y_cin,
            linewidth=2,
            color='tab:green',
            label='Modelo Cinemático')

    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)

    ax.legend()
    ax.grid(True)

In [194]:
import tensorflow as tf

R = tf.constant(0.0341, dtype=tf.float32)
L = tf.constant(0.0606, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def NumericalIntegration(Dataset, dq):
    
    init_vals = np.array([
        Dataset["Theta"].iloc[0],
    ]) 
    
    theta_cin = init_vals[0] + np.cumsum(dq[0] * dt)
    return [theta_cin]


In [195]:
def CinematicModel(Wd, We):
    dtheta_cin = (R/(2*L)) * (Wd - We)
    
    return [dtheta_cin]

In [196]:
class PINN(tf.keras.Model):

    def __init__(self, architecture, initializer):
        super(PINN, self).__init__()

        self.rnn_layers = []

        for i, units in enumerate(architecture):

            if i < len(architecture) - 1:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        return_sequences=True,
                        kernel_initializer=initializer
                    )
                )
            else:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        kernel_initializer=initializer

                    )
                )

        # camada densa final
        self.out_layer = tf.keras.layers.Dense(OUTPUT_SIZE,
                                               activation="linear",                 kernel_initializer = initializer
)

    def call(self, inputs):

        x = inputs

        for rnn in self.rnn_layers:
            x = rnn(x)

        return self.out_layer(x)

In [197]:
mean = OUT_SCALER.mean_[0]
std  = OUT_SCALER.scale_[0]
mean_tf = tf.constant(mean, dtype=tf.float32)
std_tf  = tf.constant(std, dtype=tf.float32)

In [198]:
@tf.function
def train_step(model, optimizer, x, y, Wd, We):

    with tf.GradientTape() as tape:

        pred = model(x, training=True)

        # loss dos dados
        data_loss = tf.reduce_mean(tf.square(pred - y))

        # termo físico
        physics = tf.stack(CinematicModel(Wd, We), axis=1)
        
        # normalização correta
        physics_denorm = (physics - mean_tf) / std_tf

        physics_loss = tf.reduce_mean(tf.square(pred - physics_denorm))

        loss = data_loss + physics_loss

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

In [199]:
Wd_train = tf.convert_to_tensor(Wd_train, dtype=tf.float32)
We_train = tf.convert_to_tensor(We_train, dtype=tf.float32)

x_train_tf = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.float32)

x_val_tf = tf.convert_to_tensor(x_val, dtype=tf.float32)
y_val_tf = tf.convert_to_tensor(y_val, dtype=tf.float32)


In [200]:
def EarlyStopping(model, best_loss, counter, best_weights, min_delta=1e-3):
    val_pred = model(x_val_tf)
    val_loss = tf.reduce_mean(tf.square(val_pred - y_val_tf))
    
    if val_loss < (best_loss - min_delta):
        best_loss = val_loss
        counter = 0
        best_weights = model.get_weights()

    else:
        counter += 1

    return best_loss, counter, best_weights

def TrainPINN(model, optimizer, patience=300, best_loss=np.inf):
    counter = 0
    best_weights = model.get_weights()

    for epoch in range(20000):

        loss = train_step(model, optimizer, x_train_tf, y_train_tf, Wd_train, We_train)
        best_loss, counter, best_weights =  EarlyStopping(model, best_loss, counter, best_weights)
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            model.set_weights(best_weights)
            break

        if epoch % 50 == 0:
            print(f"Epoch {epoch} Loss {loss.numpy():.6f}")

In [201]:
def GetCin(dataset): 
    dq = CinematicModel(dataset["Wd"].values, dataset["We"].values)
    q = NumericalIntegration(dataset, dq)
    return np.vstack(q).T

In [202]:
TITLES = ["train1", "train2", "test1", "test2", "test3", "val"]


def EvalModel(model):

    n_datasets = len(NormDatasets)
    n_targets = len(TARGET)

    fig, axs = plt.subplots(
        n_datasets,
        n_targets,
        figsize=(6*n_targets, 4*n_datasets)
    )

    # estrutura das métricas
    metrics = {
        name: {
            "R2_train1": [], "R2_train2": [],
            "R2_val": [],
            "R2_test1": [], "R2_test2": [], "R2_test3": [],
            "MSE_train1": [], "MSE_train2": [],
            "MSE_val": [],
            "MSE_test1": [], "MSE_test2": [], "MSE_test3": [],
        }
        for name in TARGET_INT
    }

    for i, dataset in enumerate(NormDatasets):

        x = dataset[PREDICTORS]
        y = dataset[TARGET]

        x, y = CreateSequences(x, y, TIME_STEPS)

        # predição da rede
        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()

        # desnormalização
        y_pred_diff = OUT_SCALER.inverse_transform(pred)
        y_diff = OUT_SCALER.inverse_transform(y)

        # arrays reconstruídos
        y_true = np.zeros_like(y_diff)
        y_pred = np.zeros_like(y_pred_diff)
        y_cin = GetCin(Datasets[i])
        
        n = y_true.shape[0]
        y_cin = y_cin[:n]

        # valores iniciais reais
        init_vals = np.array([
            Datasets[i][name].iloc[0] for name in TARGET_INT
        ])

        # reconstrução por integração cumulativa
        for j in range(n_targets):
            y_true[:, j] = init_vals[j] + np.cumsum(y_diff[:, j] * TS)
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred_diff[:, j] * TS)

        # cálculo das métricas
        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])

            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(
                f"{name} | {TITLES[i]} -> "
                f"R² = {r2:.4f}, MSE = {mse:.4e}"
            )

            # seleção robusta do eixo
            if n_datasets > 1 and n_targets > 1:
                ax = axs[i, j]
            elif n_datasets > 1:
                ax = axs[i]
            elif n_targets > 1:
                ax = axs[j]
            else:
                ax = axs

            PlotOut(
                ax,
                TITLES[i],
                name,
                y_true[:, j],
                y_cin[:, j],
                y_pred[:, j],
                
            )

    plt.tight_layout()
    return metrics, plt, fig

In [ ]:
from tensorflow.keras.optimizers import Adam

N_MODELS = 10  
seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

architectures = [[5], [10], [15], [20], [25],
                [5, 2], [10, 5], [15, 5], [20, 5]]

results = {}
excel_file = "resultados.xlsx"

for arch in architectures:
    for i, s in enumerate(seeds):

        tf.keras.backend.clear_session()

        init = initializers.RandomNormal(seed=int(s))

        model = PINN(architecture=arch, initializer=init)

        model.build((None, TIME_STEPS, len(PREDICTORS)))

        w0 = model.get_weights()

        opt = Adam(learning_rate=0.01)
        opt.build(model.trainable_variables)
        
        TrainPINN(model, optimizer=opt)
        
        wf = model.get_weights()

        metrics, plt, fig = EvalModel(model)
        
        arch_name = "_".join(map(str, arch))
        row = {
            "model": f"model_{arch_name}_{i}",
            "Neurons": arch,
            "W0": str([w.round(4).tolist() for w in w0]),
            "Wf": str([w.round(4).tolist() for w in wf]),
        }


        for name in TARGET_INT:
            row.update({
                f"R2_Train_1_{name}": metrics[name]["R2_train1"],
                f"MSE_Train_1_{name}": metrics[name]["MSE_train1"],
                f"R2_Train_2_{name}": metrics[name]["R2_train2"],
                f"MSE_Train_2_{name}": metrics[name]["MSE_train2"],
                f"R2_Val_{name}": metrics[name]["R2_val"],
                f"MSE_Val_{name}": metrics[name]["MSE_val"],
                f"R2_Test_1_{name}": metrics[name]["R2_test1"],
                f"MSE_Test_1_{name}": metrics[name]["MSE_test1"],
                f"R2_Test_2_{name}": metrics[name]["R2_test2"],
                f"MSE_Test_2_{name}": metrics[name]["MSE_test2"],
                f"R2_Test_3_{name}": metrics[name]["R2_test3"],
                f"MSE_Test_3_{name}": metrics[name]["MSE_test3"],
            })
        
        save_fig = any(
            any(v > 0.8 for v in metrics[name][key])
            if isinstance(metrics[name][key], (list, tuple))
            else metrics[name][key] > 0.5
            for name in TARGET_INT
            for key in metrics[name]
            if key.startswith("R2_test"))
    

        if save_fig:
            plt.savefig(
                f"./results/model_{arch_name}_{i}.pdf",
                format="pdf",
                bbox_inches="tight"
            )
            print(f"Figura salva em: ./results/model_{arch_name}_{i}.pdf")

        plt.close(fig)

        df = pd.DataFrame([row])

        # salva/atualiza Excel incrementalmente
        try:
            # tenta abrir existente e adicionar linha
            old = pd.read_excel(excel_file)
            new_df = pd.concat([old, df], ignore_index=True)
            new_df.to_excel(excel_file, index=False)
        except FileNotFoundError:
            # se não existir, cria arquivo novo
            df.to_excel(excel_file, index=False)

        print(f"Modelo {i} treinado e salvo no Excel")

Epoch 0 Loss 2.133733
Epoch 50 Loss 1.693423
Epoch 100 Loss 1.649079
Epoch 150 Loss 1.596400
Epoch 200 Loss 1.585925
Epoch 250 Loss 1.577996
Epoch 300 Loss 1.569909
Epoch 350 Loss 1.563813
Epoch 400 Loss 1.552961
Epoch 450 Loss 1.547822
Epoch 500 Loss 1.534527
Early stopping at epoch 517
Theta | train1 -> R² = 0.7717, MSE = 5.2119e-02
Theta | train2 -> R² = 0.3640, MSE = 1.4519e-01
Theta | test1 -> R² = 0.7225, MSE = 6.3749e-02
Theta | test2 -> R² = 0.2947, MSE = 1.6727e-01
Theta | test3 -> R² = -1.5064, MSE = 8.2397e-01
Theta | val -> R² = -0.2365, MSE = 4.0469e-01
Modelo 0 treinado e salvo no Excel
Epoch 0 Loss 2.130885
Epoch 50 Loss 1.639282
Epoch 100 Loss 1.579838
Epoch 150 Loss 1.569263
Epoch 200 Loss 1.555512
Epoch 250 Loss 1.536701
Epoch 300 Loss 1.513156
Epoch 350 Loss 1.488184
Early stopping at epoch 382
Theta | train1 -> R² = 0.7863, MSE = 4.8779e-02
Theta | train2 -> R² = 0.4004, MSE = 1.3690e-01
Theta | test1 -> R² = 0.6915, MSE = 7.0879e-02
Theta | test2 -> R² = 0.3382, MS